In [1]:
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from ltr_utility.dataset import DatasetName, load_by_query_dataset


In [2]:
base_path = Path("../datasets")
hold_out = (0.5, 0.2, 0.3)

datasets_to_summarize = [
    DatasetName.MQ,
    DatasetName.WEB10,
    DatasetName.YAHOO,
    DatasetName.FINDHR,
    DatasetName.FINDHRLIST,
]


In [3]:
def load_dataset_quiet(dataset_name, **kwargs):
    """Load a dataset without printing parser/cache logs in the notebook output."""
    with redirect_stdout(StringIO()):
        return load_by_query_dataset(
            base_path,
            dataset_name,
            verbose=False,
            **kwargs,
        )


def docs_per_query_stats(dataset):
    group_count = dataset.group_count
    if len(group_count) == 0:
        return np.nan, np.nan, np.nan

    return np.mean(group_count), np.min(group_count), np.max(group_count)


def target_range(dataset):
    if len(dataset.y) == 0:
        return "-"

    return f"{dataset.y.min():g} - {dataset.y.max():g}"


def summary_row(label, dataset):
    docs_mean, docs_min, docs_max = docs_per_query_stats(dataset)

    return {
        "Split": label,
        "Documents": len(dataset),
        "Query": len(dataset.unique_q),
        "Features": dataset.x.shape[1],
        "Mean docs/query": docs_mean,
        "Min docs/query": docs_min,
        "Max docs/query": docs_max,
        "Target range": target_range(dataset),
    }


def dataset_summary_table(dataset_name):
    full_dataset = load_dataset_quiet(
        dataset_name,
        get_first=None,
        min_items=None,
        max_item=None,
    )
    train, valid, test, train_valid = load_dataset_quiet(
        dataset_name,
        hold_out=hold_out,
    )

    return pd.DataFrame([
        summary_row("Full", full_dataset),
        summary_row("Train", train),
        summary_row("Validation", valid),
        summary_row("Train + valid", train_valid),
        summary_row("Test", test),
    ]).set_index("Split")


def display_dataset_summary(dataset_name):
    display(Markdown(f"### {dataset_name.name}"))
    display(
        dataset_summary_table(dataset_name).style.format({
            "Mean docs/query": "{:.2f}",
            "Min docs/query": "{:.0f}",
            "Max docs/query": "{:.0f}",
        })
    )


In [4]:
for dataset_name in datasets_to_summarize:
    display_dataset_summary(dataset_name)

### MQ

,Documents,Query,Features,Mean docs/query,Min docs/query,Max docs/query,Target range
Split,,,,,,,
Full,69623,1692,46,41.15,6,147,0 - 2
Train,10432,500,46,20.86,20,73,0 - 2
Validation,4166,500,46,8.33,8,29,0 - 2
Train + valid,14598,500,46,29.20,28,102,0 - 2
Test,6287,500,46,12.57,12,45,0 - 2


### WEB10

,Documents,Query,Features,Mean docs/query,Min docs/query,Max docs/query,Target range
Split,,,,,,,
Full,1200192,10000,136,120.02,1,908,0 - 4
Train,27732,500,136,55.46,5,185,0 - 4
Validation,10991,500,136,21.98,2,74,0 - 4
Train + valid,38723,500,136,77.45,7,259,0 - 4
Test,16996,500,136,33.99,3,111,0 - 4


### YAHOO

,Documents,Query,Features,Mean docs/query,Min docs/query,Max docs/query,Target range
Split,,,,,,,
Full,709877,29921,699,23.73,1,139,0 - 4
Train,3928,500,699,7.86,5,14,0 - 4
Validation,1462,500,699,2.92,2,6,0 - 4
Train + valid,5390,500,699,10.78,7,20,0 - 4
Test,2701,500,699,5.40,3,9,0 - 4


### FINDHR

,Documents,Query,Features,Mean docs/query,Min docs/query,Max docs/query,Target range
Split,,,,,,,
Full,500000,100,9,5000.00,5000,5000,0 - 3
Train,20000,100,9,200.00,200,200,0 - 3
Validation,8000,100,9,80.00,80,80,0 - 3
Train + valid,28000,100,9,280.00,280,280,0 - 3
Test,12000,100,9,120.00,120,120,0 - 3


### FINDHRLIST

,Documents,Query,Features,Mean docs/query,Min docs/query,Max docs/query,Target range
Split,,,,,,,
Full,500000,100,9,5000.00,5000,5000,0 - 29
Train,20000,100,9,200.00,200,200,0 - 29
Validation,8000,100,9,80.00,80,80,0 - 29
Train + valid,28000,100,9,280.00,280,280,0 - 29
Test,12000,100,9,120.00,120,120,0 - 29
